# Lesson X2: RL Evaluation - Practitioner Toolkit

## Introduction

X1 covered debugging — is anything wrong? This notebook covers the next question: **once training works, how do you measure and compare it honestly?** RL evaluation has more ways to mislead than most ML evaluation, because a single training run is one noisy sample, environments differ in what "reward" even means, and naive comparisons routinely declare a winner that a proper significance test would reject.

1. **Learning curves with confidence intervals** — across seeds, not a single run
2. **Sample-efficiency metrics** — steps-to-threshold and area-under-curve, not just final performance
3. **Episodic vs. average reward** — why the metric you pick changes which policy looks better
4. **Fair algorithm comparison** — same budget, same seeds, a real statistical test before declaring a winner

## Setup

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from scipy import stats
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

np.random.seed(42)


class EpisodeReturnLogger(BaseCallback):
    """Records (cumulative timestep, episode return) for every completed episode."""

    def __init__(self):
        super().__init__()
        self.timesteps = []
        self.returns = []

    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.timesteps.append(self.num_timesteps)
                self.returns.append(info['episode']['r'])
        return True

## Part 1: Learning Curves with Confidence Intervals

A single training run's curve is one sample from a noisy process — the same trap as X1's hyperparameter section. Proper evaluation trains multiple seeds, interpolates every run onto a common timestep grid (episodes end at different timesteps per seed, so raw curves can't be averaged directly), and reports the mean with a spread band, not a single line.

In [ ]:
N_SEEDS = 5
TIMESTEPS = 10000
GRID = np.linspace(0, TIMESTEPS, 50)

all_curves = []
for seed in range(N_SEEDS):
    env = Monitor(gym.make('CartPole-v1'))
    model = PPO('MlpPolicy', env, verbose=0, device='cpu', seed=seed)
    logger = EpisodeReturnLogger()
    model.learn(total_timesteps=TIMESTEPS, callback=logger)
    # Interpolate this seed's (timestep, return) points onto the shared grid.
    curve = np.interp(GRID, logger.timesteps, logger.returns)
    all_curves.append(curve)

all_curves = np.array(all_curves)
mean_curve = all_curves.mean(axis=0)
std_curve = all_curves.std(axis=0)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(GRID, mean_curve, color='C0', label=f'mean over {N_SEEDS} seeds')
ax.fill_between(GRID, mean_curve - std_curve, mean_curve + std_curve, alpha=0.25, color='C0',
                 label='+/- 1 std across seeds')
for curve in all_curves:
    ax.plot(GRID, curve, color='C0', alpha=0.15, linewidth=0.8)
ax.set_xlabel('Timesteps')
ax.set_ylabel('Episode return')
ax.set_title('PPO on CartPole: learning curve with cross-seed spread, not a single line')
ax.legend()
plt.tight_layout()
plt.show()

## Part 2: Sample-Efficiency Metrics

Final performance alone hides how *expensive* getting there was. Two standard sample-efficiency metrics, computed from the same 5 seeds above:

- **Steps-to-threshold**: how many environment steps until the (smoothed) return first crosses some target — directly answers "how much data does this need?"
- **Area under the curve (AUC)**: the full learning trajectory compressed into one number — rewards an algorithm that gets good *quickly*, not just one that eventually gets good

In [ ]:
THRESHOLD = 195.0  # CartPole's classic "solved" bar

steps_to_threshold = []
for curve in all_curves:
    above = np.where(curve >= THRESHOLD)[0]
    steps_to_threshold.append(GRID[above[0]] if len(above) > 0 else None)

auc = [np.trapezoid(curve, GRID) / TIMESTEPS for curve in all_curves]  # normalized to comparable units

print("Steps-to-threshold (>=195) per seed:", steps_to_threshold)
solved = [s for s in steps_to_threshold if s is not None]
if solved:
    print(f"  Mean steps-to-threshold (seeds that solved it): {np.mean(solved):.0f}")
print(f"AUC per seed (normalized): {[round(a, 1) for a in auc]}")
print(f"  Mean AUC: {np.mean(auc):.1f}")

## Part 3: Episodic vs. Average Reward

CartPole's reward is `+1` per step, so **episodic return == episode length** — a fact that quietly bakes the episode-length cap into the metric. Evaluate the *identical* trained policy under two different step caps and watch total return scale with the cap even though the policy didn't change at all. **Average reward per step** is invariant to this — the metric that actually reflects policy quality when horizon length isn't part of the task definition.

In [ ]:
def evaluate_both_metrics(model, max_steps, n_episodes=20):
    env = gym.make('CartPole-v1')
    episodic_returns, avg_rewards = [], []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=700 + ep)
        total_reward, steps = 0.0, 0
        for _ in range(max_steps):
            action, _ = model.predict(state, deterministic=True)
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            steps += 1
            if terminated or truncated:
                break
        episodic_returns.append(total_reward)
        avg_rewards.append(total_reward / steps)
    return np.mean(episodic_returns), np.mean(avg_rewards)


trained_model = PPO('MlpPolicy', gym.make('CartPole-v1'), verbose=0, device='cpu', seed=0)
trained_model.learn(total_timesteps=15000)

for cap in [100, 250, 500]:
    episodic, average = evaluate_both_metrics(trained_model, max_steps=cap)
    print(f"max_steps={cap}: episodic return={episodic:.0f}, average reward/step={average:.3f}")

print("\nEpisodic return grows with the cap even though it's the SAME policy -- comparing")
print("'episodic return' across runs with different step limits (or different environments'")
print("default horizons) silently compares horizons, not policies. Average reward/step doesn't.")

## Part 4: Fair Algorithm Comparison, With a Real Significance Test

"Algorithm A scored higher than Algorithm B" is not a finding — it's an observation that needs a test before it becomes a claim. Same environment, same total timestep budget, same 5 seeds each, then a two-sample t-test on final performance.

In [ ]:
def evaluate_policy(model, env, n_episodes=20):
    lengths = []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=900 + ep)
        for t in range(500):
            action, _ = model.predict(state, deterministic=True)
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
        lengths.append(t + 1)
    return lengths


ppo_finals, dqn_finals = [], []
for seed in range(N_SEEDS):
    ppo_env = gym.make('CartPole-v1')
    ppo_model = PPO('MlpPolicy', ppo_env, verbose=0, device='cpu', seed=seed)
    ppo_model.learn(total_timesteps=TIMESTEPS)
    ppo_finals.append(np.mean(evaluate_policy(ppo_model, ppo_env)))

    dqn_env = gym.make('CartPole-v1')
    dqn_model = DQN('MlpPolicy', dqn_env, verbose=0, device='cpu', seed=seed, learning_starts=500)
    dqn_model.learn(total_timesteps=TIMESTEPS)
    dqn_finals.append(np.mean(evaluate_policy(dqn_model, dqn_env)))

t_stat, p_value = stats.ttest_ind(ppo_finals, dqn_finals)

print(f"PPO final scores (5 seeds): {[round(s, 0) for s in ppo_finals]}, mean={np.mean(ppo_finals):.0f}")
print(f"DQN final scores (5 seeds): {[round(s, 0) for s in dqn_finals]}, mean={np.mean(dqn_finals):.0f}")
print(f"\nTwo-sample t-test: t={t_stat:.2f}, p={p_value:.5f}")
print(f"{'Statistically significant' if p_value < 0.05 else 'NOT statistically significant'} "
      f"difference at the alpha=0.05 level, given the same {TIMESTEPS}-timestep budget for both.")

## Key Takeaways

1. **Learning curves need multiple seeds and a shared timestep grid** — a single curve is a sample, not a measurement (same lesson as X1's hyperparameter section, applied to whole-algorithm comparisons)
2. **Steps-to-threshold and AUC** measure *how fast* an algorithm gets good, which final-performance-only comparisons miss entirely
3. **Episodic return silently encodes horizon length** — average reward per step is the metric that stays comparable across different step caps or episode-length conventions
4. **"Higher score" isn't a finding until a significance test says so** — this notebook's own PPO-vs-DQN comparison only becomes a real claim once the t-test's p-value backs it up
5. Every metric in this notebook was computed from the *same* training runs presented three different ways — the runs didn't change, but which one "wins" can, depending on which question you actually asked